# Sleep Stage Baseline - PyTorch

Baseline based on `ref/final-hack-iot-tcnlstm-cnn-superai-ss5.ipynb`.

The reference notebook uses 480-row windows, FFT magnitude features, `StandardScaler`, weighted F1, and a CNN-LSTM model. This version keeps that baseline design but implements the model and training loop in PyTorch.

## Setup

In [ ]:
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
WINDOW_SIZE = 480  # 30 seconds * 16 Hz
SIGNAL_COLUMNS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
LABEL_TO_ID = {"W": 0, "R": 1, "N1": 2, "N2": 3, "N3": 4}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}


def find_competition_root():
    candidates = [
        Path("/kaggle/input/super-ai-engineer-ss-6-individual-sleep-stage-classification"),
        Path("/kaggle/input/individual-sleep-stage-classification"),
        Path("/kaggle/input/Individual-Sleep-Stage-Classification"),
        Path("/kaggle/input/spai-signal-sleep-staging-classification"),
        Path("../../dataset/super-ai-engineer-ss-6-individual-sleep-stage-classification"),
        Path("../../dataset/individual-sleep-stage-classification"),
        Path("../../dataset/Individual-Sleep-Stage-Classification"),
        Path("../../dataset/sleep_stage_probe"),
    ]
    for root in candidates:
        if root.exists():
            return root
    raise FileNotFoundError("Competition data root not found. Update COMPETITION_ROOT manually.")


def first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]


COMPETITION_ROOT = find_competition_root()
TRAIN_DIR = first_existing(COMPETITION_ROOT / "train" / "train", COMPETITION_ROOT / "train")
TEST_DIR = first_existing(
    COMPETITION_ROOT / "test_segment" / "test_segment",
    COMPETITION_ROOT / "test_segment",
    COMPETITION_ROOT / "test",
)
SAMPLE_SUBMISSION = COMPETITION_ROOT / "sample_submission.csv"

print("root:", COMPETITION_ROOT)
print("train:", TRAIN_DIR)
print("test:", TEST_DIR)
print("sample submission:", SAMPLE_SUBMISSION)

## Load Files

In [ ]:
train_files = sorted(TRAIN_DIR.glob("*.csv"))
test_files = sorted(TEST_DIR.glob("**/*.csv"))

print(f"train files: {len(train_files)}")
print(f"test segment files: {len(test_files)}")

if train_files:
    print("first train file:", train_files[0])
    display(pd.read_csv(train_files[0], nrows=3))

if test_files:
    print("first test file:", test_files[0])
    display(pd.read_csv(test_files[0], nrows=3))

## Reference-Style FFT Preprocessing

In [ ]:
def clean_signal_frame(df):
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    missing = [col for col in SIGNAL_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing signal columns: {missing}")

    signals = df[SIGNAL_COLUMNS].apply(pd.to_numeric, errors="coerce")
    signals = signals.interpolate(limit_direction="both").ffill().bfill().fillna(0.0)
    return signals.astype("float32")


def majority_label(window_labels):
    counts = np.bincount(window_labels, minlength=len(LABEL_TO_ID))
    return int(counts.argmax())


def preprocess_files_with_fft(files, is_training=True, window_size=WINDOW_SIZE):
    feature_segments = []
    label_segments = []
    groups = []
    ids = []

    for file_idx, file_path in enumerate(tqdm(files, desc="Processing files")):
        df = pd.read_csv(file_path)
        signals = clean_signal_frame(df)
        num_windows = len(signals) // window_size
        if num_windows == 0:
            continue

        values = signals.iloc[: num_windows * window_size].to_numpy(dtype="float32")
        windows = values.reshape(num_windows, window_size, len(SIGNAL_COLUMNS))

        # Same feature idea as the reference notebook: FFT along each 480-row window.
        fft_windows = np.abs(np.fft.fft(windows, axis=1)).astype("float32")
        feature_segments.append(fft_windows)
        groups.extend([file_idx] * num_windows)

        if is_training:
            if "Sleep_Stage" not in df.columns:
                raise ValueError(f"Sleep_Stage missing in {file_path}")
            labels = df["Sleep_Stage"].map(LABEL_TO_ID).to_numpy()
            labels = labels[: num_windows * window_size].reshape(num_windows, window_size)
            label_segments.append(np.apply_along_axis(majority_label, axis=1, arr=labels))
        else:
            stem = file_path.stem
            ids.extend([stem] if num_windows == 1 else [f"{stem}_{i:05d}" for i in range(num_windows)])

    X = np.vstack(feature_segments).astype("float32")
    groups = np.asarray(groups)
    if is_training:
        y = np.concatenate(label_segments).astype("int64")
        return X, y, groups
    return X, ids, groups

In [ ]:
X, y, groups = preprocess_files_with_fft(train_files, is_training=True)

print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)
print("label counts:", {ID_TO_LABEL[k]: v for k, v in Counter(y).items()})

## Grouped Validation Split and Scaling

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(splitter.split(X, y, groups=groups))

X_train, X_valid = X[train_idx], X[valid_idx]
y_train, y_valid = y[train_idx], y[valid_idx]

scaler = StandardScaler()
train_shape = X_train.shape
valid_shape = X_valid.shape

X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(train_shape).astype("float32")
X_valid = scaler.transform(X_valid.reshape(-1, X_valid.shape[-1])).reshape(valid_shape).astype("float32")

print("train:", X_train.shape, Counter(y_train))
print("valid:", X_valid.shape, Counter(y_valid))

## PyTorch Datasets

In [ ]:
BATCH_SIZE = 64

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
valid_ds = TensorDataset(torch.from_numpy(X_valid), torch.from_numpy(y_valid))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)

class_counts = np.bincount(y_train, minlength=len(LABEL_TO_ID)).astype("float32")
class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = class_weights / class_weights.mean()
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("class counts:", class_counts)
print("class weights:", class_weights.detach().cpu().numpy())

## CNN-LSTM Baseline

In [ ]:
class CnnLstmBaseline(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_features, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.3),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.3),
        )
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        # x: batch, time, channels
        x = x.transpose(1, 2)
        x = self.features(x)
        x = x.transpose(1, 2)
        x, _ = self.lstm(x)
        x = x[:, -1]
        return self.classifier(x)


model = CnnLstmBaseline(n_features=X_train.shape[-1], n_classes=len(LABEL_TO_ID)).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-5,
)

model

## Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        all_preds.append(logits.detach().argmax(dim=1).cpu().numpy())
        all_targets.append(yb.detach().cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    avg_loss = total_loss / len(loader.dataset)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    return avg_loss, weighted_f1, y_true, y_pred


EPOCHS = 40
PATIENCE = 10
best_f1 = -1.0
bad_epochs = 0
best_path = Path("sleep_stage_cnn_lstm_baseline.pt")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_f1, _, _ = run_epoch(model, valid_loader, criterion)
    scheduler.step(valid_f1)

    print(
        f"epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
        f"valid_loss={valid_loss:.4f} valid_f1={valid_f1:.4f}"
    )

    if valid_f1 > best_f1:
        best_f1 = valid_f1
        bad_epochs = 0
        torch.save(model.state_dict(), best_path)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch}")
            break

print("best valid weighted F1:", best_f1)

## Validation Score

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
valid_loss, valid_f1, y_true, y_pred = run_epoch(model, valid_loader, criterion)

print("valid loss:", valid_loss)
print("valid weighted F1:", valid_f1)
print(classification_report(
    y_true,
    y_pred,
    labels=list(ID_TO_LABEL),
    target_names=[ID_TO_LABEL[i] for i in sorted(ID_TO_LABEL)],
    zero_division=0,
))

## Predict Test and Create Submission

In [ ]:
@torch.no_grad()
def predict_numpy(model, X_array, batch_size=256):
    model.eval()
    ds = TensorDataset(torch.from_numpy(X_array))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    preds = []
    for (xb,) in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        preds.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)


X_test, test_ids, _ = preprocess_files_with_fft(test_files, is_training=False)
test_shape = X_test.shape
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(test_shape).astype("float32")

test_pred = predict_numpy(model, X_test)
test_labels = [ID_TO_LABEL[int(class_id)] for class_id in test_pred]
submission = pd.DataFrame({"id": test_ids, "labels": test_labels})

if SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    submission = sample[["id"]].merge(submission, on="id", how="left")
    submission["labels"] = submission["labels"].fillna("W")

submission.to_csv("submission_cnn_lstm_baseline_pytorch.csv", index=False)
submission.head()

In [ ]:
submission["labels"].value_counts()